# Baseline Bill Passage Classification (Introduction-Only Features)

Train baseline classifiers on normalized legislative data using **only introduction-time features**.

**No downstream process leakage**: Excludes all features known only after introduction (readings, votes, committee info).

**Inputs**: `data/normalized/bills.csv`  
**Outputs**: `data/normalized/baseline_metrics_intro_only.csv`, `baseline_metrics_intro_only.json`

Evaluates 7 models across 4 splits (same structure as process-aware baseline for comparison):
- Dummy (stratified baseline)
- Logistic Regression (L1/L2, metadata-only)
- Random Forest (metadata-only)
- Logistic Regression + TF-IDF (metadata + text)
- Linear SVM + TF-IDF (metadata + text)
- Naive Bayes (text-only)

Splits:
- **Temporal US**: chronological 80/20 within US bills
- **Temporal Canada**: chronological 80/20 within Canadian bills
- **Transfer US→CA**: train on all US, test on all Canadian bills
- **Transfer CA→US**: train on all Canadian, test on all US bills

## 0. Setup and Configuration

In [1]:
import json
import os
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

# Set paths
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data"
NORM_DIR = DATA_DIR / "normalized"
INPUT_FILE = NORM_DIR / "bills.csv"
OUTPUT_DIR = NORM_DIR

print(f"ROOT: {ROOT_DIR}")
print(f"Input: {INPUT_FILE}")
print(f"Output: {OUTPUT_DIR}")

# Suppress repetitive warnings during training
warnings.filterwarnings(
    "ignore",
    message="Skipping features without any observed values",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="The max_iter was reached which means the coef_ did not converge",
)

# Global feature configuration (INTRODUCTION-ONLY: no downstream process features)
NUMERIC_COLS = [
    "year",
    "title_word_count",
    "description_word_count",
    "month_introduced",
    "parliament_number",
    "session_number",
    "reinstated",
]

CATEGORICAL_COLS = [
    "bill_type",
    "bill_type_raw",
    "chamber",
    "party",
]

TEXT_COL = "text_blob"
TARGET_COL = "passed"

ROOT: C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440
Input: C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440\data\normalized\bills.csv
Output: C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440\data\normalized


## 1. Load and Prepare Data

In [2]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare raw normalized data for modeling."""
    df = df.copy()

    # Ensure target is integer 0/1
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype(int)

    # Parse dates for temporal sorting
    if "introduced_date" in df.columns:
        df["introduced_date"] = pd.to_datetime(df["introduced_date"], errors="coerce")
    else:
        df["introduced_date"] = pd.NaT

    # Build compact text column from title + description
    title = df.get("title", "").fillna("").astype(str)
    desc = df.get("description", "").fillna("").astype(str)
    df[TEXT_COL] = (title + " " + desc).str.strip()

    # Coerce numeric columns safely when present
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Ensure categorical columns exist and are strings
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str)

    # Stable temporal key: introduced_date, then year, then original order
    df["_row_id"] = np.arange(len(df), dtype=int)
    year_fallback = pd.to_numeric(df.get("year"), errors="coerce")
    df["_sort_year"] = year_fallback.fillna(-1)

    return df

# Load and prepare data
print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
df = clean_dataframe(df)

print(f"\nLoaded {len(df):,} bills")
print(f"Sources: {df['source'].value_counts().to_dict()}")
print(f"Pass rate by source:")
print(df.groupby("source")[TARGET_COL].agg(["mean", "sum", "count"]).round(3))
print(f"\nMissing values (selected columns):")
print(df[[TARGET_COL, "introduced_date", TEXT_COL, "source"]].isnull().sum())

Loading C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440\data\normalized\bills.csv...

Loaded 120,339 bills
Sources: {'us': 113231, 'canada': 7108}
Pass rate by source:
         mean   sum   count
source                     
canada  0.150  1065    7108
us      0.021  2387  113231

Missing values (selected columns):
passed                0
introduced_date    1053
text_blob             0
source                0
dtype: int64


## 2. Build Feature Preprocessors

In [3]:
def get_metadata_preprocessor() -> ColumnTransformer:
    """Numeric imputation + scaling, categorical one-hot encoding."""
    num_pipe = Pipeline(
        steps=[
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler(with_mean=False)),
        ]
    )

    cat_pipe = Pipeline(
        steps=[
            ("impute", SimpleImputer(strategy="constant", fill_value="")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def get_metadata_text_preprocessor(max_features: int, ngram_range: tuple[int, int]) -> ColumnTransformer:
    """Combine metadata preprocessor with TF-IDF on text column."""
    metadata = get_metadata_preprocessor()

    text_pipe = Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=ngram_range,
                    min_df=5,
                    max_features=max_features,
                    stop_words="english",
                ),
            )
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("meta", metadata, NUMERIC_COLS + CATEGORICAL_COLS),
            ("text", text_pipe, TEXT_COL),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def get_text_only_preprocessor(max_features: int, ngram_range: tuple[int, int]) -> ColumnTransformer:
    """TF-IDF vectorization on text column only."""
    text_pipe = Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=ngram_range,
                    min_df=5,
                    max_features=max_features,
                    stop_words="english",
                ),
            )
        ]
    )

    return ColumnTransformer(
        transformers=[("text", text_pipe, TEXT_COL)],
        remainder="drop",
        sparse_threshold=0.3,
    )

print("Preprocessors defined.")

Preprocessors defined.


## 3. Define Baseline Models

In [4]:
def build_models(cpu_jobs: int = -1, fast_mode: bool = False, parallel_evals: bool = False) -> dict[str, Pipeline]:
    """Build 7 baseline models with configurable feature settings."""
    meta = get_metadata_preprocessor()

    # If parallelizing across model fits, keep per-model threading conservative
    inner_jobs = 1 if parallel_evals else cpu_jobs

    # Configure text vectorization based on fast_mode
    if fast_mode:
        meta_text_features = 10000
        text_only_features = 15000
        ngram_range = (1, 1)
    else:
        meta_text_features = 20000
        text_only_features = 30000
        ngram_range = (1, 2)

    meta_text = get_metadata_text_preprocessor(
        max_features=meta_text_features,
        ngram_range=ngram_range,
    )
    text_only = get_text_only_preprocessor(
        max_features=text_only_features,
        ngram_range=ngram_range,
    )

    models: dict[str, Pipeline] = {
        "dummy_stratified": Pipeline(
            steps=[
                ("prep", meta),
                ("model", DummyClassifier(strategy="stratified", random_state=42)),
            ]
        ),
        "logreg_l2_meta": Pipeline(
            steps=[
                ("prep", meta),
                (
                    "model",
                    LogisticRegression(
                        penalty="l2",
                        C=1.0,
                        solver="liblinear",
                        class_weight="balanced",
                        max_iter=2000,
                        n_jobs=inner_jobs,
                    ),
                ),
            ]
        ),
        "logreg_l1_meta": Pipeline(
            steps=[
                ("prep", meta),
                (
                    "model",
                    LogisticRegression(
                        penalty="l1",
                        C=0.5,
                        solver="liblinear",
                        class_weight="balanced",
                        max_iter=3000,
                        n_jobs=inner_jobs,
                    ),
                ),
            ]
        ),
        "random_forest_meta": Pipeline(
            steps=[
                ("prep", meta),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=300,
                        max_depth=None,
                        min_samples_leaf=2,
                        class_weight="balanced_subsample",
                        random_state=42,
                        n_jobs=inner_jobs,
                    ),
                ),
            ]
        ),
        "logreg_l2_meta_text": Pipeline(
            steps=[
                ("prep", meta_text),
                (
                    "model",
                    LogisticRegression(
                        penalty="l2",
                        C=1.0,
                        solver="saga",
                        class_weight="balanced",
                        max_iter=5000,
                        n_jobs=inner_jobs,
                    ),
                ),
            ]
        ),
        "linear_svm_meta_text": Pipeline(
            steps=[
                ("prep", meta_text),
                (
                    "model",
                    LinearSVC(
                        C=0.8,
                        class_weight="balanced",
                        max_iter=5000,
                    ),
                ),
            ]
        ),
        "naive_bayes_text": Pipeline(
            steps=[
                ("prep", text_only),
                ("model", MultinomialNB(alpha=0.5)),
            ]
        ),
    }

    return models

# Build models (using CPU job count and other settings)
cpu_jobs = max(1, os.cpu_count() or 1)
fast_mode = False  # Change to True for faster training
parallel_evals = True  # Enable parallel model evaluation

models = build_models(cpu_jobs=cpu_jobs, fast_mode=fast_mode, parallel_evals=parallel_evals)
print(f"Built {len(models)} models")
print(f"  Fast mode: {fast_mode}")
print(f"  Parallel evaluations: {parallel_evals}")

Built 7 models
  Fast mode: False
  Parallel evaluations: True


## 4. Create Temporal and Cross-Country Splits

In [5]:
@dataclass
class Split:
    """Container for train/test index split."""
    name: str
    train_idx: np.ndarray
    test_idx: np.ndarray


def make_temporal_split(df: pd.DataFrame, source: str, test_frac: float = 0.2) -> Split | None:
    """Create temporal 80/20 split within a country."""
    part = df[df["source"] == source].copy()
    if len(part) < 2:
        return None

    part = part.sort_values(["introduced_date", "_sort_year", "_row_id"])
    cut = int(len(part) * (1 - test_frac))
    cut = max(1, min(cut, len(part) - 1))

    train_idx = part.index[:cut].to_numpy()
    test_idx = part.index[cut:].to_numpy()

    return Split(name=f"temporal_{source}", train_idx=train_idx, test_idx=test_idx)


def make_cross_country_split(df: pd.DataFrame, train_source: str, test_source: str) -> Split | None:
    """Create cross-country transfer split: train on one country, test on other."""
    train_idx = df[df["source"] == train_source].index.to_numpy()
    test_idx = df[df["source"] == test_source].index.to_numpy()

    if len(train_idx) == 0 or len(test_idx) == 0:
        return None

    return Split(name=f"transfer_{train_source}_to_{test_source}", train_idx=train_idx, test_idx=test_idx)


# Create all splits
splits = [
    make_temporal_split(df, "us"),
    make_temporal_split(df, "canada"),
    make_cross_country_split(df, "us", "canada"),
    make_cross_country_split(df, "canada", "us"),
]
splits = [s for s in splits if s is not None]

print(f"Created {len(splits)} splits:")
for split in splits:
    print(f"  {split.name}: train={len(split.train_idx):,}, test={len(split.test_idx):,}")

Created 4 splits:
  temporal_us: train=90,584, test=22,647
  temporal_canada: train=5,686, test=1,422
  transfer_us_to_canada: train=113,231, test=7,108
  transfer_canada_to_us: train=7,108, test=113,231


## 5. Evaluate Models Across Splits

In [6]:
def get_score_values(model: Pipeline, x: pd.DataFrame) -> np.ndarray:
    """Extract probability scores from model."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(x)
    return model.predict(x)


def safe_metric(fn: Callable, y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Safely compute metric, returning NaN on failure."""
    try:
        value = fn(y_true, y_score)
        return float(value)
    except Exception:
        return float("nan")


def evaluate_model(model: Pipeline, x_train: pd.DataFrame, y_train: pd.Series, x_test: pd.DataFrame, y_test: pd.Series) -> dict:
    """Fit model and compute all metrics."""
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    y_score = get_score_values(model, x_test)

    return {
        "pr_auc": safe_metric(average_precision_score, y_test, y_score),
        "roc_auc": safe_metric(roc_auc_score, y_test, y_score),
        "f1": safe_metric(lambda a, b: f1_score(a, b, zero_division=0), y_test, y_pred),
        "precision": safe_metric(lambda a, b: precision_score(a, b, zero_division=0), y_test, y_pred),
        "recall": safe_metric(lambda a, b: recall_score(a, b, zero_division=0), y_test, y_pred),
        "balanced_accuracy": safe_metric(balanced_accuracy_score, y_test, y_pred),
        "predicted_positive_rate": float(np.mean(y_pred == 1)),
    }


def evaluate_model_named(
    model_name: str,
    model: Pipeline,
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_test: pd.DataFrame,
    y_test: pd.Series,
) -> dict:
    """Evaluate named model and track timing."""
    t0 = time.perf_counter()
    metrics = evaluate_model(model, x_train, y_train, x_test, y_test)
    metrics["fit_seconds"] = round(time.perf_counter() - t0, 3)
    metrics["model"] = model_name
    return metrics


def run_experiments(
    df: pd.DataFrame,
    models: dict[str, Pipeline],
    splits: list[Split],
    cpu_jobs: int = -1,
    parallel_evals: bool = False,
    use_progress_bar: bool = False,
    min_train_size: int = 500,
) -> pd.DataFrame:
    """Run all model evaluations across all splits."""
    rows: list[dict] = []
    total_evals = len(splits) * len(models)
    eval_counter = 0

    if use_progress_bar and tqdm is not None:
        progress = tqdm(total=total_evals, desc="Training baselines (intro-only)", unit="model")
    else:
        progress = None

    for split in splits:
        print(f"\n[split] {split.name}")
        x_train = df.loc[split.train_idx, NUMERIC_COLS + CATEGORICAL_COLS + [TEXT_COL]]
        y_train = df.loc[split.train_idx, TARGET_COL]
        x_test = df.loc[split.test_idx, NUMERIC_COLS + CATEGORICAL_COLS + [TEXT_COL]]
        y_test = df.loc[split.test_idx, TARGET_COL]

        if len(x_train) < min_train_size:
            print(f"  Skipped (train size {len(x_train):,} < {min_train_size:,})")
            continue

        print(
            f"  train={len(x_train):,}, test={len(x_test):,}, "
            f"train_pos={y_train.mean():.4f}, test_pos={y_test.mean():.4f}"
        )

        train_source_counts = df.loc[split.train_idx, "source"].value_counts().to_dict()
        test_source_counts = df.loc[split.test_idx, "source"].value_counts().to_dict()

        if parallel_evals:
            print(f"  evaluating {len(models)} models in parallel")
            eval_results = Parallel(n_jobs=cpu_jobs, prefer="threads")(
                delayed(evaluate_model_named)(
                    model_name,
                    model,
                    x_train,
                    y_train,
                    x_test,
                    y_test,
                )
                for model_name, model in models.items()
            )

            for metrics in eval_results:
                eval_counter += 1
                if progress is not None:
                    progress.update(1)
                else:
                    print(f"  [{eval_counter}/{total_evals}] {metrics['model']} ({metrics['fit_seconds']:.2f}s)")

                rows.append(
                    {
                        "split": split.name,
                        "model": metrics["model"],
                        "n_train": int(len(x_train)),
                        "n_test": int(len(x_test)),
                        "train_positive_rate": float(y_train.mean()),
                        "test_positive_rate": float(y_test.mean()),
                        "train_source_mix": json.dumps(train_source_counts),
                        "test_source_mix": json.dumps(test_source_counts),
                        **{k: v for k, v in metrics.items() if k != "model"},
                    }
                )
        else:
            for model_name, model in models.items():
                metrics = evaluate_model_named(model_name, model, x_train, y_train, x_test, y_test)
                eval_counter += 1
                if progress is not None:
                    progress.update(1)
                else:
                    print(f"  [{eval_counter}/{total_evals}] {model_name} ({metrics['fit_seconds']:.2f}s)")

                rows.append(
                    {
                        "split": split.name,
                        "model": model_name,
                        "n_train": int(len(x_train)),
                        "n_test": int(len(x_test)),
                        "train_positive_rate": float(y_train.mean()),
                        "test_positive_rate": float(y_test.mean()),
                        "train_source_mix": json.dumps(train_source_counts),
                        "test_source_mix": json.dumps(test_source_counts),
                        **{k: v for k, v in metrics.items() if k != "model"},
                    }
                )

    if progress is not None:
        progress.close()

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)
    result = result.sort_values(["split", "pr_auc"], ascending=[True, False]).reset_index(drop=True)
    return result

print("Evaluation functions defined.")

Evaluation functions defined.


## 6. Run Experiments and Save Results

In [7]:
print("Running introduction-only baseline experiments...")
print(f"CPU workers: {cpu_jobs}")
print(f"Fast mode: {fast_mode}")
print(f"Parallel evaluations: {parallel_evals}")
print(f"Total evaluations: {len(splits)} splits × {len(models)} models = {len(splits) * len(models)}")

metrics_df = run_experiments(
    df,
    models=models,
    splits=splits,
    cpu_jobs=cpu_jobs,
    parallel_evals=parallel_evals,
    use_progress_bar=tqdm is not None,
    min_train_size=500,
)

if metrics_df.empty:
    print("\nNo experiments were run (likely due to insufficient train size).")
else:
    print(f"\nCompleted {len(metrics_df)} evaluations")
    
    # Save results
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = OUTPUT_DIR / "baseline_metrics_intro_only.csv"
    json_path = OUTPUT_DIR / "baseline_metrics_intro_only.json"

    metrics_df.to_csv(csv_path, index=False)
    metrics_df.to_json(json_path, orient="records", indent=2)

    print(f"Saved metrics CSV:  {csv_path}")
    print(f"Saved metrics JSON: {json_path}")

    print("\n" + "="*70)
    print("Top models per split by PR-AUC (Introduction-Only)")
    print("="*70)
    for split_name, part in metrics_df.groupby("split", sort=True):
        best = part.sort_values("pr_auc", ascending=False).head(3)
        print(f"\n{split_name}")
        print(best[["model", "pr_auc", "roc_auc", "f1", "precision", "recall", "balanced_accuracy"]].to_string(index=False))

Running introduction-only baseline experiments...
CPU workers: 12
Fast mode: False
Parallel evaluations: True
Total evaluations: 4 splits × 7 models = 28


Training baselines (intro-only):   0%|          | 0/28 [00:00<?, ?model/s]


[split] temporal_us
  train=90,584, test=22,647, train_pos=0.0244, test_pos=0.0079
  evaluating 7 models in parallel


c:\Users\tamze\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



[split] temporal_canada
  train=5,686, test=1,422, train_pos=0.1769, test_pos=0.0415
  evaluating 7 models in parallel

[split] transfer_us_to_canada
  train=113,231, test=7,108, train_pos=0.0211, test_pos=0.1498
  evaluating 7 models in parallel


c:\Users\tamze\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



[split] transfer_canada_to_us
  train=7,108, test=113,231, train_pos=0.1498, test_pos=0.0211
  evaluating 7 models in parallel

Completed 28 evaluations
Saved metrics CSV:  C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440\data\normalized\baseline_metrics_intro_only.csv
Saved metrics JSON: C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440\data\normalized\baseline_metrics_intro_only.json

Top models per split by PR-AUC (Introduction-Only)

temporal_canada
              model   pr_auc  roc_auc       f1  precision   recall  balanced_accuracy
 random_forest_meta 0.713745 0.921789 0.366102   0.228814 0.915254           0.890863
logreg_l2_meta_text 0.708355 0.923984 0.092984   0.048932 0.932203           0.573952
   naive_bayes_text 0.586255 0.895134 0.537037   0.591837 0.491525           0.738426

temporal_us
               model   pr_auc  roc_auc       f1  precision   recall  balanced_accuracy
    naive_bayes_text 0.097120 0.822193 0.230576   0.210046 0.255556           0.623928
 logreg_l2_